# Homework Notebook: Download, Visualize, and Convert AdView `events.tsv` Files to FSL 3-Column EV Files

This notebook is written to run in a clean Neurodesk working directory without `git clone` or `git pull`.
It creates a local `g25/` folder, downloads the required AdView files directly from OpenNeuro `ds007486` `v1.0.0`,
visualizes one representative anatomy file and one representative AdView BOLD run inline in Jupyter, and converts
all available AdView `events.tsv` files into FEAT-ready FSL 3-column EV files.


## Outline

1. Create or reuse a local `g25/` workspace in the current directory.
2. Download the AdView files needed for this homework directly from OpenNeuro.
3. Visualize one representative anatomy file, BOLD run, and AdView event structure inline.
4. Convert all available AdView `events.tsv` files to FSL 3-column EV files.
5. Summarize exactly where the figures and EV files were written.


In [ ]:
from __future__ import annotations

import gzip
import importlib.util
import json
import ssl
import struct
import urllib.request
from pathlib import Path

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "nibabel": "nibabel",
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    raise RuntimeError(
        "This notebook expects the current Jupyter kernel to already provide: "
        + ", ".join(missing)
    )

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import Image, display

plt.rcParams["figure.dpi"] = 120
CTX = ssl._create_unverified_context()
OPENNEURO_API = "https://openneuro.org/crn/graphql"
DATASET_ID = "ds007486"
SNAPSHOT_TAG = "1.0.0"
APPROVED_SUBJECTS = ["sub-10562", "sub-10665", "sub-11028", "sub-11039", "sub-11450"]
ADVIEW_SUBJECTS_EXPECTED = ["sub-11028", "sub-11039", "sub-11450"]
REPRESENTATIVE_SUBJECT = "sub-11039"
TASK = "adview"
RUN = 1
ECHO = 1


def find_or_create_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if candidate.name == "g25":
            return candidate.resolve()
        sibling = candidate / "g25"
        if sibling.exists() and sibling.is_dir():
            return sibling.resolve()
    project_root = start / "g25"
    project_root.mkdir(parents=True, exist_ok=True)
    return project_root.resolve()


def gql(query: str, variables: dict) -> dict:
    payload = json.dumps({"query": query, "variables": variables}).encode("utf-8")
    req = urllib.request.Request(
        OPENNEURO_API,
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, context=CTX) as resp:
        return json.load(resp)


def snapshot_files(tree: str | None = None) -> list[dict]:
    query = '''
    query($datasetId: ID!, $tag: String!, $tree: String) {
      snapshot(datasetId: $datasetId, tag: $tag) {
        files(tree: $tree) {
          key
          filename
          directory
          urls
          size
        }
      }
    }
    '''
    return gql(query, {"datasetId": DATASET_ID, "tag": SNAPSHOT_TAG, "tree": tree})["data"]["snapshot"]["files"]


def subject_directory_key(subject: str, root_rows: list[dict]) -> str:
    for row in root_rows:
        if row.get("directory") and row["filename"] == subject:
            return row["key"]
    raise FileNotFoundError(f"Missing subject directory for {subject}")


def child_directory_key(parent_key: str, dirname: str) -> str:
    rows = snapshot_files(parent_key)
    for row in rows:
        if row.get("directory") and row["filename"] == dirname:
            return row["key"]
    raise FileNotFoundError(f"Missing child directory {dirname} under {parent_key}")


def file_url_map(tree_key: str) -> dict[str, str]:
    rows = snapshot_files(tree_key)
    return {row["filename"]: row["urls"][0] for row in rows if row.get("urls")}


def is_valid_nifti_gz(path: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        with gzip.open(path, "rb") as src:
            header = src.read(352)
            for _chunk in iter(lambda: src.read(1024 * 1024), b""):
                pass
    except OSError:
        return False
    if len(header) < 352:
        return False
    sizeof_hdr_le = struct.unpack("<I", header[:4])[0]
    sizeof_hdr_be = struct.unpack(">I", header[:4])[0]
    return sizeof_hdr_le == 348 or sizeof_hdr_be == 348


def download_url(url: str, destination: Path, validate_nifti: bool = False) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0:
        if validate_nifti and not is_valid_nifti_gz(destination):
            destination.unlink()
        else:
            return destination

    with urllib.request.urlopen(url, context=CTX) as resp:
        destination.write_bytes(resp.read())

    if destination.stat().st_size == 0:
        raise RuntimeError(f"Downloaded empty file for {destination.name}")
    if validate_nifti and not is_valid_nifti_gz(destination):
        raise RuntimeError(f"Downloaded invalid gzipped NIfTI file: {destination}")
    return destination


def write_if_missing(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        path.write_text(text)


def patch_task_metadata(path: Path) -> None:
    if not path.exists():
        return
    payload = json.loads(path.read_text())
    if str(payload.get("TaskName", "")).startswith("TODO") or not payload.get("TaskName"):
        payload["TaskName"] = "Passive Ad Viewing"
    path.write_text(json.dumps(payload, indent=2) + "\n")


def plot_orthogonal_slices(volume: np.ndarray, title: str, out_path: Path, cmap: str = "gray") -> None:
    volume = np.asarray(volume)
    x, y, z = [dim // 2 for dim in volume.shape[:3]]
    vmax = float(np.percentile(volume, 99))
    vmin = float(np.percentile(volume, 1))
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(np.rot90(volume[x, :, :]), cmap=cmap, vmin=vmin, vmax=vmax)
    axes[0].set_title("Sagittal")
    axes[1].imshow(np.rot90(volume[:, y, :]), cmap=cmap, vmin=vmin, vmax=vmax)
    axes[1].set_title("Coronal")
    axes[2].imshow(np.rot90(volume[:, :, z]), cmap=cmap, vmin=vmin, vmax=vmax)
    axes[2].set_title("Axial")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)


def derive_adview_trial_type(row: pd.Series) -> str:
    response = str(row.get("attention_response", "")).strip()
    is_attention = str(row.get("is_attention_check", "")).strip().lower() in {"1", "true", "yes"}
    if response == "no_response":
        return "missed_trial"
    if is_attention:
        return "attention_check"
    return str(row["trial_type"])


def plot_adview_events(events: pd.DataFrame, count_path: Path, timeline_path: Path) -> None:
    order = ["Every_day_products", "Gambling", "vmPFC", "attention_check", "missed_trial"]
    counts = events["derived_trial_type"].value_counts().reindex(order, fill_value=0)

    fig, ax = plt.subplots(figsize=(8, 4))
    counts.plot(kind="bar", color=["#5B8FF9", "#F4664A", "#5D7092", "#61DDAA", "#9A60B4"], ax=ax)
    ax.set_title("AdView trial counts by derived event label")
    ax.set_xlabel("Derived trial type")
    ax.set_ylabel("Number of events")
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()
    fig.savefig(count_path, dpi=150)
    plt.close(fig)

    y_map = {label: idx for idx, label in enumerate(order)}
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.scatter(events["onset"], [y_map[label] for label in events["derived_trial_type"]], s=28, c="#4C78A8")
    ax.set_yticks(list(y_map.values()))
    ax.set_yticklabels(order)
    ax.set_xlabel("Onset (s)")
    ax.set_title("AdView event timeline for the representative run")
    ax.grid(alpha=0.25, axis="x")
    fig.tight_layout()
    fig.savefig(timeline_path, dpi=150)
    plt.close(fig)


def convert_adview_events_to_evs(events_path: Path, ev_dir: Path, run_id: int = 1) -> list[dict[str, object]]:
    ev_dir.mkdir(parents=True, exist_ok=True)
    frame = pd.read_csv(events_path, sep="\t").copy()
    frame["derived_trial_type"] = frame.apply(derive_adview_trial_type, axis=1)
    labels = ["Every_day_products", "Gambling", "vmPFC", "attention_check", "missed_trial"]
    manifest = []
    for label in labels:
        subset = frame.loc[frame["derived_trial_type"] == label, ["onset", "duration"]].copy()
        subset["amplitude"] = 1.0
        out_path = ev_dir / f"run-{run_id}_{label}.txt"
        if label == "missed_trial" and subset.empty:
            if out_path.exists():
                out_path.unlink()
            continue
        with out_path.open("w") as stream:
            for row in subset.itertuples(index=False):
                stream.write(f"{float(row.onset):.6f}\t{float(row.duration):.6f}\t1.0\n")
        manifest.append({"event": label, "rows": len(subset), "path": str(out_path)})
    return manifest


PROJECT_ROOT = find_or_create_project_root()
BIDS_ROOT = PROJECT_ROOT / "bids"
EV_ROOT = PROJECT_ROOT / "derivatives" / "fsl" / "EVfiles"
OUTPUT_DIR = PROJECT_ROOT / "output" / "jupyter-notebook"
CACHE_DIR = OUTPUT_DIR / "_cache"
FIG_DIR = OUTPUT_DIR / "figures"

for path in [BIDS_ROOT, EV_ROOT, OUTPUT_DIR, CACHE_DIR, FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

write_if_missing(
    PROJECT_ROOT / "README.md",
    "# G25\n\nStandalone AdView homework workspace created by this notebook.\n",
)

print(f"Project root: {PROJECT_ROOT}")
print(f"BIDS root: {BIDS_ROOT}")
print(f"EV root: {EV_ROOT}")
print(f"Output dir: {OUTPUT_DIR}")


## Step 1 - Create a standalone `g25/` workspace and download AdView files

Purpose:
- Run this notebook independently, so it cannot assume an existing repo checkout.
- This step creates the minimal `g25/` folder structure and downloads the files needed for the rest of the homework.

What the code is doing:
- Query the `ds007486` `v1.0.0` snapshot on OpenNeuro.
- Detect which approved subjects actually have `adview` in the snapshot.
- Download the root metadata files and `task-adview_bold.json`.
- Download `adview` `events.tsv` and `events.json` for every subject that has AdView.
- Download one representative T1w image and one representative AdView BOLD run for `sub-11039`.

QC checkpoint:
- The notebook should report a local `g25/` folder.
- The AdView availability table should show `sub-11028`, `sub-11039`, and `sub-11450`.
- The representative anatomy, BOLD, and event files should exist locally after this cell runs.


In [ ]:
root_rows = snapshot_files()
root_url_map = {row["filename"]: row["urls"][0] for row in root_rows if row.get("urls")}

for filename in ["README", "dataset_description.json", "participants.tsv", "participants.json", "task-adview_bold.json"]:
    if filename in root_url_map:
        download_url(root_url_map[filename], BIDS_ROOT / filename, validate_nifti=False)
patch_task_metadata(BIDS_ROOT / "task-adview_bold.json")

adview_rows = []
subject_maps = {}
available_adview_subjects = []

for subject in APPROVED_SUBJECTS:
    subject_key = subject_directory_key(subject, root_rows)
    anat_key = child_directory_key(subject_key, "anat")
    func_key = child_directory_key(subject_key, "func")
    anat_map = file_url_map(anat_key)
    func_map = file_url_map(func_key)
    subject_maps[subject] = {"anat": anat_map, "func": func_map}

    events_tsv = f"{subject}_task-{TASK}_run-{RUN}_events.tsv"
    events_json = f"{subject}_task-{TASK}_run-{RUN}_events.json"
    has_adview = events_tsv in func_map
    adview_rows.append({"subject": subject, "has_adview": has_adview, "events_file": events_tsv if has_adview else ""})

    if has_adview:
        available_adview_subjects.append(subject)
        subject_func_dir = BIDS_ROOT / subject / "func"
        download_url(func_map[events_tsv], subject_func_dir / events_tsv, validate_nifti=False)
        if events_json in func_map:
            download_url(func_map[events_json], subject_func_dir / events_json, validate_nifti=False)

anat_filename = f"{REPRESENTATIVE_SUBJECT}_T1w.nii.gz"
bold_filename = f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_echo-{ECHO}_part-mag_bold.nii.gz"
bold_json_filename = f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_echo-{ECHO}_part-mag_bold.json"
rep_cache_dir = CACHE_DIR / "download-visualize" / REPRESENTATIVE_SUBJECT
rep_cache_dir.mkdir(parents=True, exist_ok=True)

anat_path = download_url(subject_maps[REPRESENTATIVE_SUBJECT]["anat"][anat_filename], rep_cache_dir / anat_filename, validate_nifti=True)
bold_path = download_url(subject_maps[REPRESENTATIVE_SUBJECT]["func"][bold_filename], rep_cache_dir / bold_filename, validate_nifti=True)
bold_json_path = download_url(subject_maps[REPRESENTATIVE_SUBJECT]["func"][bold_json_filename], rep_cache_dir / bold_json_filename, validate_nifti=False)
events_path = BIDS_ROOT / REPRESENTATIVE_SUBJECT / "func" / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_events.tsv"

available_subjects_csv = OUTPUT_DIR / "g25-adview-available-subjects.csv"
workspace_summary_path = OUTPUT_DIR / "g25-adview-homework-workspace-summary.json"
pd.DataFrame(adview_rows).to_csv(available_subjects_csv, index=False)
workspace_summary_path.write_text(
    json.dumps(
        {
            "project_root": str(PROJECT_ROOT),
            "dataset_id": DATASET_ID,
            "snapshot_tag": SNAPSHOT_TAG,
            "available_adview_subjects": available_adview_subjects,
            "representative_subject": REPRESENTATIVE_SUBJECT,
        },
        indent=2,
    )
    + "\n"
)

display(pd.DataFrame(adview_rows))
print(f"Representative T1w: {anat_path}")
print(f"Representative AdView BOLD: {bold_path}")
print(f"Representative AdView events: {events_path}")
print(f"Available subjects CSV: {available_subjects_csv}")
print(f"Workspace summary JSON: {workspace_summary_path}")

assert available_adview_subjects == ADVIEW_SUBJECTS_EXPECTED
assert anat_path.exists()
assert bold_path.exists()
assert bold_json_path.exists()
assert events_path.exists()


## Step 2 - Visualize the representative anatomy, AdView BOLD run, and AdView events

Purpose:
- This is the main homework requirement: show that the data can be downloaded and visualized in a Jupyter notebook.

What the code is doing:
- Load the representative T1w and AdView BOLD files with `nibabel`.
- Save one anatomy figure and one middle-volume BOLD figure.
- Load the representative AdView `events.tsv`.
- Derive the event labels that will later become EV files.
- Save an event-count plot and a timeline plot.

QC checkpoint:
- The T1w image should be 3D.
- The AdView BOLD image should be 4D.
- The event plots and image figures should display inline below the cell.


In [ ]:
t1_img = nib.load(str(anat_path))
bold_img = nib.load(str(bold_path))
middle_bold_idx = bold_img.shape[3] // 2
middle_bold = np.asarray(bold_img.dataobj[..., middle_bold_idx])

events = pd.read_csv(events_path, sep="\t").copy()
events["derived_trial_type"] = events.apply(derive_adview_trial_type, axis=1)

t1_png = FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_T1w.png"
bold_png = FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_middle-volume.png"
event_count_png = FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_event-counts.png"
event_timeline_png = FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_event-timeline.png"

plot_orthogonal_slices(np.asarray(t1_img.dataobj), f"{REPRESENTATIVE_SUBJECT} T1w", t1_png)
plot_orthogonal_slices(middle_bold, f"{REPRESENTATIVE_SUBJECT} AdView run-{RUN} middle volume", bold_png)
plot_adview_events(events, event_count_png, event_timeline_png)

print(f"T1w shape: {t1_img.shape}")
print(f"AdView BOLD shape: {bold_img.shape}")
print(f"Representative events rows: {len(events)}")
print(events['derived_trial_type'].value_counts().to_string())

display(events.head(12))
display(Image(filename=str(t1_png)))
display(Image(filename=str(bold_png)))
display(Image(filename=str(event_count_png)))
display(Image(filename=str(event_timeline_png)))

assert len(t1_img.shape) == 3
assert len(bold_img.shape) == 4
assert t1_png.exists()
assert bold_png.exists()
assert event_count_png.exists()
assert event_timeline_png.exists()


## Step 3 - Convert all available AdView `events.tsv` files to FSL 3-column EV files

Purpose:
- This is the extra-credit portion.
- The goal is to show that the AdView BIDS event tables can be converted into FEAT-ready EV text files for every available AdView subject.

What the code is doing:
- Loop over every subject with AdView data in `v1.0.0`.
- Read the local `events.tsv` file for that subject.
- Derive event labels from `trial_type`, `is_attention_check`, and `attention_response`.
- Write FSL 3-column EV files into `g25/derivatives/fsl/EVfiles/sub-<id>/adview/`.
- Save a manifest and a per-subject summary CSV.

QC checkpoint:
- The summary should include `sub-11028`, `sub-11039`, and `sub-11450`.
- Each EV file should have exactly three columns.
- `sub-11039` should include a `missed_trial` EV file.


In [ ]:
ev_manifest_rows = []
run_summary_rows = []

for subject in ADVIEW_SUBJECTS_EXPECTED:
    subject_events_path = BIDS_ROOT / subject / "func" / f"{subject}_task-{TASK}_run-{RUN}_events.tsv"
    subject_ev_dir = EV_ROOT / subject / TASK
    manifest_rows = convert_adview_events_to_evs(subject_events_path, subject_ev_dir, run_id=RUN)
    run_summary_rows.append({"subject": subject, "run": RUN, "n_ev_files": len(manifest_rows)})
    for row in manifest_rows:
        ev_manifest_rows.append(
            {
                "subject": subject,
                "run": RUN,
                "event": row["event"],
                "rows": row["rows"],
                "path": row["path"],
            }
        )

run_summary = pd.DataFrame(run_summary_rows).sort_values(["subject", "run"]).reset_index(drop=True)
ev_manifest = pd.DataFrame(ev_manifest_rows).sort_values(["subject", "run", "event"]).reset_index(drop=True)

run_summary_path = OUTPUT_DIR / "g25-adview-run-summary.csv"
ev_manifest_path = OUTPUT_DIR / "g25-adview-ev-manifest.csv"
run_summary.to_csv(run_summary_path, index=False)
ev_manifest.to_csv(ev_manifest_path, index=False)

print(f"Run summary CSV: {run_summary_path}")
print(f"EV manifest CSV: {ev_manifest_path}")
display(run_summary)
display(ev_manifest)

preview_tables = []
for ev_path in sorted((EV_ROOT / REPRESENTATIVE_SUBJECT / TASK).glob("run-1_*.txt")):
    table = pd.read_csv(ev_path, sep=r"\s+", header=None, names=["onset", "duration", "amplitude"])
    table.insert(0, "file", ev_path.name)
    preview_tables.append(table.head())

if preview_tables:
    display(pd.concat(preview_tables, ignore_index=True))

assert set(run_summary["subject"]) == set(ADVIEW_SUBJECTS_EXPECTED)
assert (EV_ROOT / "sub-11039" / TASK / "run-1_missed_trial.txt").exists()
assert all(pd.read_csv(Path(path), sep=r"\s+", header=None).shape[1] == 3 for path in ev_manifest["path"])


## Final summary

This last cell lists the important outputs so the grader can see exactly what the notebook created.


In [ ]:
summary = pd.DataFrame(
    [
        {"artifact": "Workspace root", "path": str(PROJECT_ROOT)},
        {"artifact": "Available subjects CSV", "path": str(OUTPUT_DIR / "g25-adview-available-subjects.csv")},
        {"artifact": "Workspace summary JSON", "path": str(OUTPUT_DIR / "g25-adview-homework-workspace-summary.json")},
        {"artifact": "Representative T1w PNG", "path": str(FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_T1w.png")},
        {"artifact": "Representative AdView BOLD PNG", "path": str(FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_middle-volume.png")},
        {"artifact": "AdView event count PNG", "path": str(FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_event-counts.png")},
        {"artifact": "AdView event timeline PNG", "path": str(FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_event-timeline.png")},
        {"artifact": "All-subject AdView EV summary", "path": str(OUTPUT_DIR / "g25-adview-run-summary.csv")},
        {"artifact": "All-subject AdView EV manifest", "path": str(OUTPUT_DIR / "g25-adview-ev-manifest.csv")},
        {"artifact": "All-subject AdView EV root", "path": str(EV_ROOT)},
    ]
)
display(summary)


## Troubleshooting

- This notebook should create `./g25` automatically if it is run from a clean directory.
- It uses only relative paths inside the new workspace.
- It does not require `git clone`, `git pull`, FSL, or MRIcroGL.
- It does require internet access so it can download data directly from OpenNeuro.
